# A3-M2-2 · 데이터와 임베딩
교안의 같은 번호 단원과 함께 봅니다. 노트북은 01 → 02 → 03 순서로 실행합니다.

## 01-1. CSV 읽어서 눈으로 보기

In [1]:
import pandas as pd
faq = pd.read_csv("shop_faq.csv")
tests = pd.read_csv("shop_faq_tests.csv")
print("FAQ:", faq.shape, "| 평가 질문:", tests.shape)
print(faq.dtypes.to_string())
print()
print(faq.head(5).to_string(index=False))
print()
print(faq["category"].value_counts().to_string())

FAQ: (20, 4) | 평가 질문: (12, 2)
faq_id      int64
category      str
question      str
answer        str

 faq_id category                 question                                                   answer
    101       배송              배송은 언제 오나요?           주문 후 평일 기준 2~3일 이내에 도착합니다. 제주·도서산간은 3~5일 걸립니다.
    102       배송              배송비는 얼마인가요?           30000원 이상 구매 시 무료 배송이며, 미만이면 배송비 3000원이 부과됩니다.
    103       배송  주문한 상품의 배송 조회는 어디서 하나요?            마이페이지 > 주문 내역에서 운송장 번호를 눌러 배송 위치를 확인할 수 있습니다.
    104       배송 냉장고 같은 대형 상품은 어떻게 배송되나요? 무게 30kg 이상 또는 가로 150cm 이상 상품은 화물로 배송되며 설치 일정은 전화로 협의합니다.
    105       배송              해외로도 배송되나요?                        현재는 국내 배송만 가능하며 해외 배송은 지원하지 않습니다.

category
배송      5
교환반품    4
결제      3
환불      2
주문      2
회원      2
적립금     2


## 01-2. 검색에 쓸 문장(doc) 만들기

In [2]:
faq["doc"] = faq["question"] + " " + faq["answer"]
print(faq.loc[0, "doc"])
print("doc 글자 수:", faq["doc"].str.len().describe()[["min", "mean", "max"]].round(1).to_dict())

배송은 언제 오나요? 주문 후 평일 기준 2~3일 이내에 도착합니다. 제주·도서산간은 3~5일 걸립니다.
doc 글자 수: {'min': 45.0, 'mean': 59.2, 'max': 81.0}


## 01-3. Hugging Face 파이프라인 출력 들여다보기

In [3]:
import numpy as np
import faiss
from transformers import pipeline
from emb import EMB_MODEL

fe = pipeline("feature-extraction", model=EMB_MODEL, device=-1)
s = "query: 택배 언제 와요?"
out = fe(s)
arr = np.array(out)
print("모델:", EMB_MODEL)
print("출력 모양:", arr.shape, "← 문장 1개 × 토큰 수 × 384")
print("토큰:", fe.tokenizer.tokenize(s))
v = arr[0].mean(axis=0)
print("토큰 평균:", v.shape, "길이", round(float(np.linalg.norm(v)), 3))

c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--intfloat--multilingual-e5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python

모델: intfloat/multilingual-e5-small
출력 모양: (1, 12, 384) ← 문장 1개 × 토큰 수 × 384
토큰: ['▁que', 'ry', ':', '▁', '택', '배', '▁언제', '▁와', '요', '?']
토큰 평균: (384,) 길이 4.499


## 01-4. embed() 함수 만들고 FAQ 20건을 벡터로

In [4]:
def embed(texts, kind="passage"):
    outs = fe([f"{kind}: {t}" for t in texts])                       # 문장마다 [1][토큰][384]
    X = np.array([np.array(o[0]).mean(axis=0) for o in outs], dtype="float32")
    faiss.normalize_L2(X)                                              # 길이 1 → 내적 = 코사인
    return X

X = embed(faq["doc"].tolist())
print("X:", X.shape, X.dtype)
print("길이:", np.linalg.norm(X, axis=1).round(3)[:5], "...")
np.save("faq_vectors.npy", X)
print("저장: faq_vectors.npy")

X: (20, 384) float32
길이: [1. 1. 1. 1. 1.] ...
저장: faq_vectors.npy


## 01-5. FAISS 전에 — 질문 하나와 FAQ 20건 코사인 직접 계산

In [5]:
q = embed(["택배 언제 와요?"], kind="query")
sims = X @ q[0]                                   # (20, 384) @ (384,) → (20,)
top = np.argsort(-sims)[:5]
print(faq.loc[top, ["faq_id", "category", "question"]].assign(cosine=sims[top].round(3)).to_string(index=False))

 faq_id category                question  cosine
    101       배송             배송은 언제 오나요?   0.846
    301     교환반품      교환이나 반품은 언제까지 되나요?   0.827
    103       배송 주문한 상품의 배송 조회는 어디서 하나요?   0.824
    502       주문     주문 후 배송지를 바꿀 수 있나요?   0.822
    105       배송             해외로도 배송되나요?   0.819
